In [0]:
--  Gold Delivery Risk Features

CREATE OR REPLACE TABLE gold_delivery_risk AS
SELECT
  product_id,
  product_name,
  category_name,
  department_name,
  market,
  order_region,
  customer_segment,
  shipping_mode,
  payment_type,

  -- Target Variable (what we want to predict)
  is_late_delivery,
  is_canceled,
  is_early_delivery,

  -- Delivery Features
  scheduled_ship_days,
  actual_ship_days,
  delivery_delay,

  -- Financial Features
  sales,
  order_qty,
  unit_price,
  discount_rate,
  profit_per_order,
  profit_ratio,
  is_profitable,

  -- Inventory Features
  current_stock,
  reorder_point,
  safety_stock,
  stock_buffer,
  stock_status,
  stockout_risk,
  order_now_flag,

  -- Time Features
  order_date,
  ship_date,
  MONTH(order_date)                     AS order_month,
  DAYOFWEEK(order_date)                 AS order_day_of_week,
  YEAR(order_date)                      AS order_year,
  QUARTER(order_date)                   AS order_quarter

FROM silver_supply_chain
WHERE order_date IS NOT NULL;



SELECT * FROM gold_delivery_risk
WHERE current_stock IS NOT NULL
AND reorder_point IS NOT NULL;




SELECT *,
  CASE
    WHEN defect_rate > 3.0 AND inspection_result = 'Fail' THEN 'High Risk'
    WHEN defect_rate > 1.5 OR inspection_result = 'Pending' THEN 'Medium Risk'
    ELSE 'Low Risk'
  END AS supplier_risk_score
FROM gold_supplier_risk;


----------------------------------------------------------------

-- Gold Demand Forecast Features

CREATE OR REPLACE TABLE gold_demand_forecast AS
SELECT
  s.product_id,
  s.inventory_date,
  MONTH(s.inventory_date)               AS month,
  YEAR(s.inventory_date)                AS year,
  QUARTER(s.inventory_date)             AS quarter,
  DAYOFWEEK(s.inventory_date)           AS day_of_week,
  s.store_id,
  s.category,
  s.region,

  -- Target Variable
  s.units_sold,

  -- Inventory Features
  s.inventory_level,
  s.units_ordered,
  s.stock_after_sales,
  s.order_fulfillment_gap,
  s.is_stockout,
  s.inventory_health,

  -- Supplier Features
  sc.supplier_name,
  sc.supplier_lead_time,
  sc.defect_rate,
  sc.defect_risk_level,
  sc.shipping_cost,
  sc.transport_mode,
  sc.production_volume,
  sc.manufacturing_cost,
  sc.is_defective,
  sc.stock_health

FROM silver_store_inventory s
LEFT JOIN silver_supply_chain_data sc
  ON s.product_id = sc.sku;




CREATE OR REPLACE TABLE gold_demand_forecast AS
SELECT *,
  CASE 
    WHEN month IN (11, 12) THEN 1 
    ELSE 0 
  END AS is_holiday_season,
  CASE 
    WHEN day_of_week IN (1, 7) THEN 1 
    ELSE 0 
  END AS is_weekend
FROM gold_demand_forecast;




---------------------------------------------------------
--  Gold Supplier Risk Features

CREATE OR REPLACE TABLE gold_supplier_risk AS
SELECT
  sku,
  product_type,
  supplier_name,
  supplier_location,
  transport_mode,
  route,
  shipping_carrier,

  -- Target Variable
  is_defective,
  defect_risk_level,
  inspection_result,

  -- Risk Features
  defect_rate,
  lead_time,
  supplier_lead_time,
  manufacturing_lead_time,
  shipping_time,
  shipping_cost,
  shipping_cost_per_unit,

  -- Production Features
  production_volume,
  manufacturing_cost,
  order_quantity,
  stock_level,
  stock_health,
  availability,

  -- Financial Features
  price,
  revenue,
  revenue_per_unit,
  products_sold,
  total_cost

FROM silver_supply_chain_data;




-------------------------------------------------

-- all features combined

CREATE OR REPLACE TABLE gold_ml_master AS
SELECT
  d.product_id,
  d.product_name,
  d.category_name,
  d.market,
  d.order_region,
  d.customer_segment,
  d.shipping_mode,
  d.payment_type,
  d.order_month,
  d.order_year,
  d.order_quarter,
  d.order_day_of_week,

  -- Delivery Features
  d.scheduled_ship_days,
  d.actual_ship_days,
  d.delivery_delay,
  d.is_late_delivery,
  d.is_canceled,
  d.is_early_delivery,

  -- Financial Features
  d.sales,
  d.order_qty,
  d.unit_price,
  d.discount_rate,
  d.profit_per_order,
  d.profit_ratio,
  d.is_profitable,

  -- Inventory Features
  d.current_stock,
  d.reorder_point,
  d.safety_stock,
  d.stock_buffer,
  d.stock_status,
  d.stockout_risk,
  d.order_now_flag,

  -- Supplier Features
  sc.supplier_name,
  sc.supplier_lead_time,
  sc.defect_rate,
  sc.defect_risk_level,
  sc.shipping_cost,
  sc.transport_mode,
  sc.manufacturing_cost,
  sc.is_defective,
  sc.revenue_per_unit,
  sc.shipping_cost_per_unit

FROM gold_delivery_risk d
LEFT JOIN silver_supply_chain_data sc
  ON CAST(d.product_id AS STRING) = sc.sku;




-------------------------------------------------

-- verifying the gold layer

SELECT 'gold_delivery_risk'   AS table_name, COUNT(*) AS rows FROM gold_delivery_risk
UNION ALL
SELECT 'gold_demand_forecast' AS table_name, COUNT(*) AS rows FROM gold_demand_forecast
UNION ALL
SELECT 'gold_supplier_risk'   AS table_name, COUNT(*) AS rows FROM gold_supplier_risk
UNION ALL
SELECT 'gold_ml_master'       AS table_name, COUNT(*) AS rows FROM gold_ml_master;




SELECT * FROM gold_delivery_risk LIMIT 2;




SELECT * FROM gold_delivery_risk;

SELECT * FROM gold_demand_forecast;

SELECT * FROM gold_supplier_risk;

SELECT * FROM gold_ml_master;



-- Check product ID formats in each table
(SELECT 'silver_supply_chain' AS tbl, CAST(product_id AS STRING) AS product_id FROM silver_supply_chain LIMIT 5)
UNION ALL
(SELECT 'silver_store_inventory' AS tbl, product_id FROM silver_store_inventory LIMIT 5);

SELECT 'silver_supply_chain_data' AS tbl, sku AS product_id FROM silver_supply_chain_data LIMIT 5;

-- Check if category names overlap between the two tables
SELECT DISTINCT category_name FROM silver_supply_chain LIMIT 10;

SELECT DISTINCT product_type FROM silver_supply_chain_data LIMIT 10;

-- Check store inventory product IDs
SELECT DISTINCT product_id FROM silver_store_inventory LIMIT 10;

-- Check supply chain SKUs
SELECT DISTINCT sku FROM silver_supply_chain_data LIMIT 10;